# 03_base_master — Dataset para Modelagem de Churn

Constrói o dataset de modelagem a partir dos parquets gerados nos notebooks anteriores.

**Estratégia temporal:**
- `CUTOFF_TREINO`: última data usada para features de treino
- Features: comportamento do cliente **antes** do CUTOFF
- Target `churn_h3`: o cliente pediu algo nos `HORIZONTE_MESES` meses após o CUTOFF?
  - `0` → pediu (ativo)
  - `1` → não pediu (churnou)

| Input | Origem | Features |
|---|---|---|
| `cliente_mes.parquet` | notebook 01 | volume, valor, regularidade mensal |
| `cliente_fidelidade.parquet` | notebook 01 | padrão de intervalos entre compras |
| `cliente_item_tendencia.parquet` | notebook 02 | tendência de queda por item |
| Banco | `base_clientes_enr.sql` | NOME, DIASINADIMPLENTE |

**Output:** `data/processed/df_model.parquet`

In [1]:
import sys, os, warnings
from pathlib import Path
import pandas as pd
import numpy as np
warnings.filterwarnings("ignore")

PROJECT_ROOT = next(p for p in [Path.cwd()] + list(Path.cwd().parents) if (p / "src").exists())
sys.path.insert(0, str(PROJECT_ROOT))
os.chdir(PROJECT_ROOT)

from src import db
from src.config import (
    DATA_REF, INICIO,
    CUTOFF_TREINO, CUTOFF_TESTE, HORIZONTE_MESES,
    PROCESSED_DIR, QUERIES_DIR,
)

anos = sorted({INICIO[:4], str(CUTOFF_TREINO.year), str(CUTOFF_TESTE.year), str(DATA_REF.year)})
linha = "|"
for ano in anos:
    linha += f"─── {ano} ───|"

ct  = CUTOFF_TREINO.strftime("%d/%m/%y")
cte = CUTOFF_TESTE.strftime("%d/%m/%y")
dr  = DATA_REF.strftime("%d/%m/%y")

print(linha)
print(f"             ↑            ↑             ↑")
print(f"      CUTOFF_TREINO  CUTOFF_TESTE   DATA_REF")
print(f"        ({ct})   ({cte})   ({dr})")
print()
print(f"  Features treino : {INICIO[:7]} → {CUTOFF_TREINO.strftime('%Y-%m')}")
print(f"  Outcome treino  : {(CUTOFF_TREINO + pd.DateOffset(months=1)).strftime('%Y-%m')} → {(CUTOFF_TREINO + pd.DateOffset(months=HORIZONTE_MESES)).strftime('%Y-%m')} ({HORIZONTE_MESES} meses)")
print(f"  Features teste  : {INICIO[:7]} → {CUTOFF_TESTE.strftime('%Y-%m')}")
print(f"  Outcome teste   : {(CUTOFF_TESTE + pd.DateOffset(months=1)).strftime('%Y-%m')} → {(CUTOFF_TESTE + pd.DateOffset(months=HORIZONTE_MESES)).strftime('%Y-%m')} ({HORIZONTE_MESES} meses)")

|─── 2023 ───|─── 2024 ───|─── 2025 ───|─── 2026 ───|
             ↑            ↑             ↑
      CUTOFF_TREINO  CUTOFF_TESTE   DATA_REF
        (31/12/24)   (30/11/25)   (01/05/26)

  Features treino : 2023-01 → 2024-12
  Outcome treino  : 2025-01 → 2025-03 (3 meses)
  Features teste  : 2023-01 → 2025-11
  Outcome teste   : 2025-12 → 2026-02 (3 meses)


---
## Seção 1 — Carregar dados

In [2]:
cm  = pd.read_parquet(PROCESSED_DIR / "cliente_mes.parquet")
cf  = pd.read_parquet(PROCESSED_DIR / "cliente_fidelidade.parquet")
cit = pd.read_parquet(PROCESSED_DIR / "cliente_item_tendencia.parquet")

# ANO_MES pode ser salvo como string em parquet — garantir Period
cm["ANO_MES"] = cm["ANO_MES"].astype(str).pipe(lambda s: pd.PeriodIndex(s, freq="M"))

print(f"cliente_mes            : {cm.shape}")
print(f"cliente_fidelidade     : {cf.shape}")
print(f"cliente_item_tendencia : {cit.shape}")

cliente_mes            : (9980, 9)
cliente_fidelidade     : (765, 15)
cliente_item_tendencia : (12967, 9)


In [3]:
sql = Path(QUERIES_DIR / "base_clientes.sql").read_text()
df_clientes_raw = db.get_data(sql)

print("Colunas disponíveis:", list(df_clientes_raw.columns))

Colunas disponíveis: ['CODIGO', 'NOME', 'CLASSE', 'CIDADEENTREGA', 'UFENTREGA', 'ATIVIDADE', 'STATUS', 'CODCONDICAOPAGAMENTO', 'CONDICAOPAGAMENTO', 'DIASINADIMPLENTE']


In [4]:
df_perfil = (
    df_clientes_raw[["CODIGO", "NOME", "DIASINADIMPLENTE"]]
    .drop_duplicates(subset="CODIGO")
    .rename(columns={"CODIGO": "CLIENTE"})
)

print(f"Perfil de clientes: {df_perfil.shape}")

Perfil de clientes: (1153, 3)


---
## Seção 2 — Janela temporal e target

O CUTOFF é o fim da janela de features. O target é calculado verificando se o
cliente pediu algo **dentro da janela de tolerância da sua categoria**.

Janelas personalizadas (em meses após o CUTOFF):

| Categoria | Janela de outcome | Lógica |
|---|---|---|
| premium | 2 meses | compra frequente — sumiço curto já é sinal |
| alto | 2 meses | compra frequente — sumiço curto já é sinal |
| medio | 3 meses | frequência média |
| baixo | 6 meses | compra esporádica — tolera silêncio maior |

| Label | Significado |
|---|---|
| `churn = 0` | cliente pediu dentro da sua janela — ativo |
| `churn = 1` | cliente NÃO pediu dentro da sua janela — churnou |

In [5]:
cutoff_period = pd.Period(CUTOFF_TREINO, "M")  # 2024-12

# Janela de features: meses ATÉ o CUTOFF (inclusive)
cm_feat = cm[cm["ANO_MES"] <= cutoff_period].copy()

# Clientes elegíveis: tiveram pelo menos 1 pedido antes do CUTOFF
clientes_elegiveis = cm_feat["CLIENTE"].unique()

# Threshold de tolerância por categoria (meses de silêncio permitidos)
THRESHOLD_CHURN = {"baixo": 6, "medio": 3, "alto": 2, "premium": 2}

# Categoria predominante de cada cliente na janela pré-CUTOFF
categoria_pre = (
    cm_feat.groupby("CLIENTE")["categoria_pedido"]
    .agg(lambda x: x.mode()[0])
    .rename("categoria")
    .reset_index()
)

# Primeiro pedido pós-CUTOFF de cada cliente (se existir)
cm_pos = cm[cm["ANO_MES"] > cutoff_period][["CLIENTE", "ANO_MES"]].copy()
primeiro_pos = (
    cm_pos.groupby("CLIENTE")["ANO_MES"]
    .min()
    .reset_index()
    .rename(columns={"ANO_MES": "primeiro_pedido_pos"})
)

# Montar df_target com threshold personalizado por cliente
df_target = pd.DataFrame({"CLIENTE": clientes_elegiveis})
df_target = df_target.merge(categoria_pre, on="CLIENTE", how="left")
df_target["threshold_meses"] = df_target["categoria"].map(THRESHOLD_CHURN)
df_target = df_target.merge(primeiro_pos, on="CLIENTE", how="left")

# Janela de outcome personalizada: CUTOFF + threshold da categoria
df_target["outcome_end"] = df_target["threshold_meses"].apply(
    lambda t: cutoff_period + t
)

# churn = 1 se não pediu nada após o CUTOFF
#         OU pediu depois da janela permitida para a categoria
df_target["churn"] = (
    df_target["primeiro_pedido_pos"].isna() |
    (df_target["primeiro_pedido_pos"] > df_target["outcome_end"])
).astype(int)

print(f"Janela features : {INICIO} → {cutoff_period}")
print(f"Janela outcome  : personalizada por categoria")
print(f"  baixo   → até {cutoff_period + 6}")
print(f"  medio   → até {cutoff_period + 3}")
print(f"  alto    → até {cutoff_period + 2}")
print(f"  premium → até {cutoff_period + 2}")
print()
print(f"Clientes elegíveis: {len(df_target)}")
print(f"Churn  (churn=1): {df_target['churn'].sum()} ({df_target['churn'].mean():.1%})")
print(f"Ativo  (churn=0): {(df_target['churn']==0).sum()} ({(df_target['churn']==0).mean():.1%})")
print()
print("Por categoria:")
print(
    df_target.groupby("categoria")["churn"]
    .agg(n="count", churnou="sum", taxa="mean")
    .assign(taxa=lambda x: x["taxa"].map("{:.1%}".format))
)

Janela features : 2023-01-01 → 2024-12
Janela outcome  : personalizada por categoria
  baixo   → até 2025-06
  medio   → até 2025-03
  alto    → até 2025-02
  premium → até 2025-02

Clientes elegíveis: 610
Churn  (churn=1): 221 (36.2%)
Ativo  (churn=0): 389 (63.8%)

Por categoria:
             n  churnou   taxa
categoria                     
alto        55       13  23.6%
baixo      494      200  40.5%
medio       46        5  10.9%
premium     15        3  20.0%


---
## Seção 3 — Features de comportamento mensal

Agregamos `cliente_mes` no nível cliente usando **apenas a janela de features** (pré-CUTOFF).

| Feature | Descrição |
|---|---|
| `n_meses_ativos` | Meses com pelo menos 1 pedido na janela |
| `total_pedidos` | Total de pedidos únicos na janela |
| `total_valor` | Faturamento total na janela |
| `media_pedidos_mes` | Média de pedidos por mês |
| `cv_pedidos` | Coeficiente de variação — regularidade (maior = mais irregular) |
| `ticket_medio` | Ticket médio na janela |
| `itens_por_pedido` | Média de itens por pedido |
| `categoria_pedido` | Categoria predominante (moda) |
| `meses_sem_pedido_pre` | Meses de silêncio até o CUTOFF |

In [ ]:
def meses_desde(serie_periodo: pd.Series, ref: pd.Period) -> pd.Series:
    return serie_periodo.apply(
        lambda x: (ref.year - x.year) * 12 + (ref.month - x.month)
    )

# Janela total de meses disponível (INICIO até CUTOFF)
janela_meses = (
    (cutoff_period.year - pd.Period(INICIO, "M").year) * 12 +
    (cutoff_period.month - pd.Period(INICIO, "M").month)
)

features_comportamento = (
    cm_feat.groupby("CLIENTE")
    .agg(
        n_meses_ativos    = ("ANO_MES",         "count"),
        ultimo_mes_pre    = ("ANO_MES",          "max"),
        total_pedidos     = ("total_pedidos",    "sum"),
        total_valor       = ("total_valor",      "sum"),
        media_pedidos_mes = ("total_pedidos",    "mean"),
        std_pedidos_mes   = ("total_pedidos",    "std"),
        ticket_medio      = ("ticket_medio",     "mean"),
        itens_por_pedido  = ("itens_por_pedido", "mean"),
        categoria_pedido  = ("categoria_pedido", lambda x: x.mode()[0]),
    )
    .reset_index()
)

features_comportamento["cv_pedidos"] = (
    features_comportamento["std_pedidos_mes"] /
    features_comportamento["media_pedidos_mes"]
)
# NaN preservado — indefinido para n_meses_ativos==1; preenchido em s6_impute após sem_historico_cadencia

features_comportamento["meses_sem_pedido_pre"] = meses_desde(
    features_comportamento["ultimo_mes_pre"], cutoff_period
)

# razao_atividade: proporção de meses ativos na janela
# substitui a bifurcação baixo_recorrente/sazonal — o modelo aprende o limiar
features_comportamento["razao_atividade"] = (
    features_comportamento["n_meses_ativos"] / janela_meses
)

features_comportamento = features_comportamento.drop(
    columns=["ultimo_mes_pre", "std_pedidos_mes"]
)

print(f"Janela: {janela_meses} meses ({INICIO} → {cutoff_period})")
print(f"Shape: {features_comportamento.shape}")
display(features_comportamento.describe().round(2))

---
## Seção 4 — Features de fidelidade

Padrão de intervalos entre compras — recalculado sobre **cm_feat (pré-CUTOFF)** para evitar leakage.

| Feature | Descrição |
|---|---|
| `intervalo_medio` | Média de meses entre pedidos consecutivos na janela |
| `max_intervalo` | Maior intervalo registrado na janela |
| `n_intervalos` | Quantidade de intervalos (proxy de senioridade) |
| `categoria_cliente` | Moda de categoria_pedido na janela |

In [ ]:
def calc_fidelidade_pre_cutoff(grupo):
    meses = sorted(grupo["ANO_MES"].tolist())
    categoria = grupo["categoria_pedido"].mode().iloc[0]
    if len(meses) < 2:
        return pd.Series({
            "intervalo_medio":   np.nan,
            "max_intervalo":     np.nan,   # indefinido — não existe intervalo para 1 mês ativo
            "n_intervalos":      0,
            "categoria_cliente": categoria,
        })
    intervalos = [meses[i].ordinal - meses[i-1].ordinal for i in range(1, len(meses))]
    return pd.Series({
        "intervalo_medio":   np.mean(intervalos),
        "max_intervalo":     max(intervalos),
        "n_intervalos":      len(intervalos),
        "categoria_cliente": categoria,
    })

features_fidelidade = (
    cm_feat
    .sort_values(["CLIENTE", "ANO_MES"])
    .groupby("CLIENTE")
    .apply(calc_fidelidade_pre_cutoff, include_groups=False)
    .reset_index()
)

print(f"Shape: {features_fidelidade.shape}")
display(features_fidelidade.describe().round(2))

---
## Seção 5 — Features de tendência de itens

Captura o sinal de migração gradual para concorrente: queda progressiva no volume
de itens específicos antes do churn.

> **Leakage residual:** `tendencia_slope` foi calculado sobre todo o período (2023–DATA_REF).
> Influência dos meses pós-CUTOFF é pequena (regressão linear sobre 2+ anos).
> `var_pct_ultimo` foi **excluído** — representa o último mês dos dados (muito posterior ao CUTOFF).

In [8]:
features_itens = (
    cit.groupby("CLIENTE")
    .agg(
        slope_portfolio_medio = ("tendencia_slope", "mean"),
        pct_itens_queda       = ("tendencia_slope", lambda x: (x < 0).sum() / len(x)),
        n_itens_portfolio     = ("SERVICO",         "count"),
    )
    .reset_index()
)

print(f"Shape: {features_itens.shape}")
display(features_itens.describe().round(3))

Shape: (764, 4)


,CLIENTE,slope_portfolio_medio,pct_itens_queda,n_itens_portfolio
count,764.000,602.000,764.000,764.000
mean,788.045,-0.025,0.350,16.973
std,513.853,0.292,0.287,29.790
min,1.000,-4.000,0.000,1.000
25%,291.250,-0.036,0.000,3.000
50%,801.500,-0.000,0.374,9.000
75%,1248.250,0.019,0.500,18.000
max,1674.000,2.000,1.000,512.000


---
## Seção 6 — Montar df_model

Join sequencial partindo do target (clientes elegíveis como âncora).

> **Atenção:** do `df_target` usamos **apenas `CLIENTE` e `churn`** — as demais colunas
> (`primeiro_pedido_pos`, `outcome_end`, `categoria`, `threshold_meses`) são artefatos
> intermediários da construção do target calculados com dados **pós-CUTOFF** e não podem
> entrar como features. A Seção 8 (teste) já seguia esse padrão corretamente.

In [ ]:
df_model = (
    df_target[["CLIENTE", "churn"]]   # apenas identificador + target — sem colunas pós-CUTOFF
    .merge(features_comportamento, on="CLIENTE", how="left")
    .merge(features_fidelidade,    on="CLIENTE", how="left")
    .merge(features_itens,         on="CLIENTE", how="left")
    .merge(df_perfil,              on="CLIENTE", how="left")
)

print(f"Shape final: {df_model.shape}")
print(f"Colunas    : {list(df_model.columns)}")

missing = df_model.isna().sum()
missing = missing[missing > 0]
if len(missing):
    print(f"\nMissing values:")
    print(missing)
else:
    print("\nSem missing values.")

In [10]:
print("=== Target ===")
vc = df_model["churn"].value_counts().rename({0: "Ativo", 1: "Churnou"})
print(vc)
print(f"Taxa de churn: {df_model['churn'].mean():.1%}")

print("\n=== Por categoria ===")
print(
    df_model.groupby("categoria_cliente")["churn"]
    .agg(n="count", churnou="sum", taxa="mean")
    .assign(taxa=lambda x: x["taxa"].map("{:.1%}".format))
)

=== Target ===
churn
Ativo      389
Churnou    221
Name: count, dtype: int64
Taxa de churn: 36.2%

=== Por categoria ===
                     n  churnou   taxa
categoria_cliente                     
alto                55       13  23.6%
baixo              494      200  40.5%
medio               46        5  10.9%
premium             15        3  20.0%


In [ ]:
# ── Flags — criados ANTES de qualquer fillna ──────────────────────────────

# Eixo cadência: n_meses_ativos==1 → cv/intervalo estruturalmente indefinidos
df_model["sem_historico_cadencia"] = (df_model["n_intervalos"] == 0).astype(int)

# Eixo itens: ausência total OU todos os slopes NaN em cit
df_model["sem_historico_itens"] = (
    df_model["pct_itens_queda"].isna() | df_model["slope_portfolio_medio"].isna()
).astype(int)

# ── Medianas de cadência — pré-fillna, só sobre clientes com histórico real ──
# Três medianas independentes, uma por feature.
# Compartilhar um único sentinel entre features amarra todas à flag → VIF inflado.
# Mesma disciplina do eixo de itens (sem_historico_itens VIF ~2.7 — já funcionando).

mask_com_hist = df_model["sem_historico_cadencia"] == 0

medianas_cadencia_treino = {
    "cv_pedidos"     : df_model.loc[mask_com_hist, "cv_pedidos"].median(),
    "intervalo_medio": df_model.loc[mask_com_hist, "intervalo_medio"].median(),
    "max_intervalo"  : df_model.loc[mask_com_hist, "max_intervalo"].median(),
}

# ── Imputação cadência ─────────────────────────────────────────────────────
for feat, med in medianas_cadencia_treino.items():
    df_model[feat] = df_model[feat].fillna(med)

# ── Imputação itens ────────────────────────────────────────────────────────
df_model["slope_portfolio_medio"] = df_model["slope_portfolio_medio"].fillna(0)
df_model["pct_itens_queda"]       = df_model["pct_itens_queda"].fillna(0)

# ── Relatório ──────────────────────────────────────────────────────────────
n_cad  = df_model["sem_historico_cadencia"].sum()
n_item = df_model["sem_historico_itens"].sum()
print(f"sem_historico_cadencia : {n_cad} clientes imputados ({n_cad/len(df_model):.1%})")
print(f"sem_historico_itens    : {n_item} clientes imputados ({n_item/len(df_model):.1%})")
print()
print("Medianas de cadência (treino | sem_historico_cadencia == 0):")
for feat, med in medianas_cadencia_treino.items():
    print(f"  {feat:<20}: {med:.4f}")

# Sanity check físico
assert medianas_cadencia_treino["max_intervalo"] >= medianas_cadencia_treino["intervalo_medio"], \
    "FALHA: mediana max_intervalo < mediana intervalo_medio"
print(f"\n✓ max_intervalo ({medianas_cadencia_treino['max_intervalo']:.4f}) >= intervalo_medio ({medianas_cadencia_treino['intervalo_medio']:.4f})")

missing = df_model.isna().sum()
missing = missing[missing > 0]
print("\nMissing restantes:", "nenhum" if missing.empty else missing.to_string())

---
## Seção 7 — Salvar

Output: `data/processed/df_model.parquet` — input do `04_modelo.ipynb`.

In [ ]:
df_model.to_parquet(PROCESSED_DIR / "df_model_treino.parquet", index=False)

# medianas_cadencia_treino (dict) definido em s6_impute
# Seção 8 aplica exatamente esses valores no teste — sem recálculo (anti-vazamento)

print("=== Salvo ===")
print(f"  {PROCESSED_DIR / 'df_model_treino.parquet'}")
print(f"  Shape  : {df_model.shape}")
print(f"  Colunas: {list(df_model.columns)}")
print(f"  Churn  : {df_model['churn'].sum()} ({df_model['churn'].mean():.1%})")

---
## Seção 8 — Dataset de Teste (CUTOFF_TESTE)

Repete a construção usando `CUTOFF_TESTE = dez/2025` para validação temporal.

**Limitação aceita:** clientes "baixo" teriam janela de outcome até jun/2026,
mas `DATA_REF = mai/2026` — falta 1 mês. A janela é travada em `DATA_REF`.

In [ ]:
cutoff_period_teste = pd.Period(CUTOFF_TESTE, "M")         # 2025-12
data_ref_period     = pd.Period(DATA_REF, "M")              # 2026-05

# ── Features pré-CUTOFF_TESTE ──────────────────────────────────────────────
cm_feat_teste = cm[cm["ANO_MES"] <= cutoff_period_teste].copy()
clientes_elegiveis_teste = cm_feat_teste["CLIENTE"].unique()

janela_meses_teste = (
    (cutoff_period_teste.year - pd.Period(INICIO, "M").year) * 12 +
    (cutoff_period_teste.month - pd.Period(INICIO, "M").month)
)

# ── Target personalizado com outcome_end travado em DATA_REF ───────────────
categoria_pre_teste = (
    cm_feat_teste.groupby("CLIENTE")["categoria_pedido"]
    .agg(lambda x: x.mode()[0])
    .rename("categoria")
    .reset_index()
)

cm_pos_teste = cm[cm["ANO_MES"] > cutoff_period_teste][["CLIENTE", "ANO_MES"]].copy()
primeiro_pos_teste = (
    cm_pos_teste.groupby("CLIENTE")["ANO_MES"]
    .min()
    .reset_index()
    .rename(columns={"ANO_MES": "primeiro_pedido_pos"})
)

df_target_teste = pd.DataFrame({"CLIENTE": clientes_elegiveis_teste})
df_target_teste = df_target_teste.merge(categoria_pre_teste, on="CLIENTE", how="left")
df_target_teste["threshold_meses"] = df_target_teste["categoria"].map(THRESHOLD_CHURN)
df_target_teste = df_target_teste.merge(primeiro_pos_teste, on="CLIENTE", how="left")

df_target_teste["outcome_end"] = df_target_teste["threshold_meses"].apply(
    lambda t: min(cutoff_period_teste + t, data_ref_period)
)

df_target_teste["churn"] = (
    df_target_teste["primeiro_pedido_pos"].isna() |
    (df_target_teste["primeiro_pedido_pos"] > df_target_teste["outcome_end"])
).astype(int)

# ── Features de comportamento ──────────────────────────────────────────────
features_comportamento_teste = (
    cm_feat_teste.groupby("CLIENTE")
    .agg(
        n_meses_ativos    = ("ANO_MES",         "count"),
        ultimo_mes_pre    = ("ANO_MES",          "max"),
        total_pedidos     = ("total_pedidos",    "sum"),
        total_valor       = ("total_valor",      "sum"),
        media_pedidos_mes = ("total_pedidos",    "mean"),
        std_pedidos_mes   = ("total_pedidos",    "std"),
        ticket_medio      = ("ticket_medio",     "mean"),
        itens_por_pedido  = ("itens_por_pedido", "mean"),
        categoria_pedido  = ("categoria_pedido", lambda x: x.mode()[0]),
    )
    .reset_index()
)

features_comportamento_teste["cv_pedidos"] = (
    features_comportamento_teste["std_pedidos_mes"] /
    features_comportamento_teste["media_pedidos_mes"]
)
# NaN preservado — preenchido abaixo com mediana do treino após criação de sem_historico_cadencia

features_comportamento_teste["meses_sem_pedido_pre"] = meses_desde(
    features_comportamento_teste["ultimo_mes_pre"], cutoff_period_teste
)

features_comportamento_teste["razao_atividade"] = (
    features_comportamento_teste["n_meses_ativos"] / janela_meses_teste
)

features_comportamento_teste = features_comportamento_teste.drop(
    columns=["ultimo_mes_pre", "std_pedidos_mes"]
)

# ── Features de fidelidade ─────────────────────────────────────────────────
features_fidelidade_teste = (
    cm_feat_teste
    .sort_values(["CLIENTE", "ANO_MES"])
    .groupby("CLIENTE")
    .apply(calc_fidelidade_pre_cutoff, include_groups=False)
    .reset_index()
)

# ── Montar df_model_teste ──────────────────────────────────────────────────
df_model_teste = (
    df_target_teste[["CLIENTE", "churn"]]
    .merge(features_comportamento_teste, on="CLIENTE", how="left")
    .merge(features_fidelidade_teste,    on="CLIENTE", how="left")
    .merge(features_itens,               on="CLIENTE", how="left")
    .merge(df_perfil,                    on="CLIENTE", how="left")
)

# ── Flags — criados ANTES de qualquer fillna ───────────────────────────────
df_model_teste["sem_historico_cadencia"] = (df_model_teste["n_intervalos"] == 0).astype(int)

df_model_teste["sem_historico_itens"] = (
    df_model_teste["pct_itens_queda"].isna() | df_model_teste["slope_portfolio_medio"].isna()
).astype(int)

# ── Imputação cadência — medianas do TREINO, sem recálculo ────────────────
for feat, med in medianas_cadencia_treino.items():
    df_model_teste[feat] = df_model_teste[feat].fillna(med)

# ── Imputação itens ────────────────────────────────────────────────────────
df_model_teste["slope_portfolio_medio"] = df_model_teste["slope_portfolio_medio"].fillna(0)
df_model_teste["pct_itens_queda"]       = df_model_teste["pct_itens_queda"].fillna(0)

n_cad_t  = df_model_teste["sem_historico_cadencia"].sum()
n_item_t = df_model_teste["sem_historico_itens"].sum()
print(f"Janela features : {INICIO} → {cutoff_period_teste}")
print(f"Clientes        : {len(df_model_teste)}")
print(f"Churn           : {df_model_teste['churn'].sum()} ({df_model_teste['churn'].mean():.1%})")
print(f"sem_historico_cadencia : {n_cad_t} clientes imputados ({n_cad_t/len(df_model_teste):.1%})")
print(f"sem_historico_itens    : {n_item_t} clientes imputados ({n_item_t/len(df_model_teste):.1%})")
print()
print("Por categoria:")
print(
    df_model_teste.groupby("categoria_cliente")["churn"]
    .agg(n="count", churnou="sum", taxa="mean")
    .assign(taxa=lambda x: x["taxa"].map("{:.1%}".format))
)

In [14]:
df_model_teste.to_parquet(PROCESSED_DIR / "df_model_teste.parquet", index=False)

print("=== Salvo ===")
print(f"  {PROCESSED_DIR / 'df_model_teste.parquet'}")
print(f"  Shape  : {df_model_teste.shape}")
print(f"  Churn  : {df_model_teste['churn'].sum()} ({df_model_teste['churn'].mean():.1%})")

=== Salvo ===
  G:\Meu Drive\central_eto\data\processed\df_model_teste.parquet
  Shape  : (724, 21)
  Churn  : 355 (49.0%)


---
## Seção 9 — Investigação: por que o churn do teste é maior?

In [15]:
print("=" * 55)
print("PERGUNTA 1 — Clientes novos de 2025 no teste")
print("=" * 55)

clientes_treino = set(df_model["CLIENTE"])
clientes_teste  = set(df_model_teste["CLIENTE"])

novos_2025 = clientes_teste - clientes_treino
so_treino  = clientes_treino - clientes_teste
em_ambos   = clientes_treino & clientes_teste

print(f"Clientes só no treino (saíram antes de 2025) : {len(so_treino)}")
print(f"Clientes em ambos                           : {len(em_ambos)}")
print(f"Clientes novos no teste (entraram em 2025)  : {len(novos_2025)}")

churn_novos = df_model_teste[df_model_teste["CLIENTE"].isin(novos_2025)]["churn"].mean()
churn_antigos = df_model_teste[df_model_teste["CLIENTE"].isin(em_ambos)]["churn"].mean()
print(f"\nTaxa de churn — clientes novos de 2025 : {churn_novos:.1%}")
print(f"Taxa de churn — clientes em ambos      : {churn_antigos:.1%}")

print()
print("=" * 55)
print("PERGUNTA 2 — Churn por categoria (treino vs teste)")
print("=" * 55)

comp_treino = (
    df_model.groupby("categoria_cliente")["churn"]
    .agg(n="count", taxa="mean")
    .assign(taxa=lambda x: x["taxa"].map("{:.1%}".format))
    .rename(columns={"n": "n_treino", "taxa": "churn_treino"})
)
comp_teste = (
    df_model_teste.groupby("categoria_cliente")["churn"]
    .agg(n="count", taxa="mean")
    .assign(taxa=lambda x: x["taxa"].map("{:.1%}".format))
    .rename(columns={"n": "n_teste", "taxa": "churn_teste"})
)
print(comp_treino.join(comp_teste).to_string())

print()
print("=" * 55)
print("PERGUNTA 3 — Impacto do DATA_REF nos clientes 'baixo'")
print("=" * 55)

baixo_teste = df_target_teste[df_target_teste["categoria"] == "baixo"].copy()
travados = (baixo_teste["outcome_end"] == data_ref_period).sum()
total_baixo = len(baixo_teste)
churn_baixo_teste = baixo_teste["churn"].mean()

print(f"Clientes 'baixo' no teste           : {total_baixo}")
print(f"Com outcome_end travado em mai/2026 : {travados} ({travados/total_baixo:.1%})")
print(f"Taxa de churn dos 'baixo' no teste  : {churn_baixo_teste:.1%}")
print(f"Taxa de churn dos 'baixo' no treino : {df_model[df_model['categoria_cliente']=='baixo']['churn'].mean():.1%}")

PERGUNTA 1 — Clientes novos de 2025 no teste
Clientes só no treino (saíram antes de 2025) : 0
Clientes em ambos                           : 610
Clientes novos no teste (entraram em 2025)  : 114

Taxa de churn — clientes novos de 2025 : 41.2%
Taxa de churn — clientes em ambos      : 50.5%

PERGUNTA 2 — Churn por categoria (treino vs teste)
                   n_treino churn_treino  n_teste churn_teste
categoria_cliente                                            
alto                     55        23.6%       61       36.1%
baixo                   494        40.5%      592       53.7%
medio                    46        10.9%       55       20.0%
premium                  15        20.0%       16       25.0%

PERGUNTA 3 — Impacto do DATA_REF nos clientes 'baixo'
Clientes 'baixo' no teste           : 592
Com outcome_end travado em mai/2026 : 592 (100.0%)
Taxa de churn dos 'baixo' no teste  : 53.7%
Taxa de churn dos 'baixo' no treino : 40.5%
